## Monte-carlo Pairs Trading

For this model, I want the model to take into account N crypto currencies and re-balance a portfolio for some frequency based on a pairs trading strategy. I want the strategy to dynamically search over all the currencies and group them into pairs if they are correlated by a certain amount, disregarding pairs that have the lowest correlation.




$$

$$

In [ ]:
import datetime as dt

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

# Parameters
N_sims = 50
N_days = 365
mu_leader = 0.3
mu_follower = 0.3
sigma_leader = 0.05
sigma_follower = 0.05
S_initial = 100

# Time grid
start = dt.datetime.now()
end = start + dt.timedelta(days=N_days)
T = np.arange(start, end, np.timedelta64(1, "1m"))
delta_t = 1 / len(T)

# Initialize the price processes
S_leader = S_initial * np.ones((N_sims, len(T)))
S_follower = S_initial * np.ones((N_sims, len(T)))

# Correlated Brownian motions
rho = 0.1  # Correlation between leader and follower
B_leader = np.random.normal(0, np.sqrt(delta_t), (N_sims, len(T)))
W_leader = B_leader
W_follower = rho * B_leader + np.sqrt(1 - rho**2) * np.random.normal(
    0, np.sqrt(delta_t), (N_sims, len(T))
)

# Initial conditions
S_initial_leader = 100
S_initial_follower = 100

# Simulate the model
for i in tqdm(range(1, len(T))):
    S_leader[:, i] = (
        S_leader[:, i - 1]
        + sigma_leader * W_leader[:, i] * S_leader[:, i - 1]
        + mu_leader * delta_t * S_leader[:, i - 1]
    )
    S_follower[:, i] = (
        S_follower[:, i - 1]
        + sigma_follower * W_follower[:, i] * S_follower[:, i - 1]
        + mu_follower * delta_t * S_follower[:, i - 1]
    )

# Plot the results
size = 2
fig, axes = plt.subplots(size, size, figsize=(15, 12))
fig.suptitle(f"{size}x{size} Plot Grid", fontsize=16, fontweight="bold")
for i in range(size):
    for j in range(size):
        axes[i, j].plot(T, S_leader[i * size + j, :])
        axes[i, j].plot(T, S_follower[i * size + j, :])
        axes[i, j].legend(["S_leader", "S_follower"])
        axes[i, j].set_xlabel("Time")
        axes[i, j].set_ylabel("Price")
        axes[i, j].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        axes[i, j].xaxis.set_major_locator(mdates.DayLocator(interval=int(N_days / 10)))
        axes[i, j].tick_params(axis="x", rotation=45)
plt.show()

In [ ]:
S_spread = 2 * np.abs(S_leader - S_follower) / (S_leader + S_follower)

size = 2
fig, axes = plt.subplots(size, size, figsize=(15, 12))
fig.suptitle(f"{size}x{size} Plot Grid", fontsize=16, fontweight="bold")
for i in range(size):
    for j in range(size):
        axes[i, j].plot(T, S_spread[i * size + j, :])
        axes[i, j].legend(["S_spread"])
        axes[i, j].set_xlabel("Time")
        axes[i, j].set_ylabel("Spread")
        axes[i, j].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        axes[i, j].xaxis.set_major_locator(mdates.DayLocator(interval=int(N_days / 10)))
        axes[i, j].tick_params(axis="x", rotation=45)
plt.show()

In [ ]:
# Create kernel for convolution
# 30 days with minutely resolution: 30 * 24 * 60 = 43,200 minutes
window = 30 * 24 * 60
kernel = np.ones(window) / window

# Apply convolution to each row
S_mean = np.apply_along_axis(
    lambda x: np.convolve(x, kernel, mode="valid"), axis=1, arr=S_spread
)

# Fill the beginning with expanding means
# for i in range(S_spread.shape[0]):  # For each row
#     for j in range(window - 1):  # For each position before full window
#         if j == 0:
#             # First position: just the first value
#             S_mean[i, j] = S_spread[i, j]
#         else:
#             # Other positions: mean of all available data points
#             S_mean[i, j] = np.mean(S_spread[i, : j + 1])

$$
\sigma = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (x_i - \mu)^2}
$$

In [ ]:
# Create kernel for convolution
# 30 days with minutely resolution: 30 * 24 * 60 = 43,200 minutes
window = 30 * 24 * 60
kernel = np.ones(window) / window

# Apply convolution to each row
S_std_to_squared = np.apply_along_axis(
    lambda x: np.convolve(x, kernel, mode="valid"), axis=1, arr=S_spread**2
)

# Fill the beginning with expanding means
# for i in range(S_std_to_squared.shape[0]):  # For each row
#     for j in range(window - 1):  # For each position before full window
#         if j == 0:
#             # First position: just the first value
#             S_std_to_squared[i, j] = S_mean_squared[i, j]
#         else:
#             # Other positions: mean of all available data points
#             S_std_to_squared[i, j] = np.mean(S_mean_squared[i, : j + 1])

# Get the variance
S_var = S_std_to_squared - S_mean**2

# Get the square root of the variance
S_std = np.sqrt(np.maximum(S_var, 0))

In [ ]:
S_std

In [ ]:
S_z_score = (S_spread[:, window - 1 :] - S_mean) / S_std

In [ ]:
S_z_score

In [ ]:
np.sign(S_z_score) * np.minimum(np.abs(S_z_score), 3)